# Assignment 7

## Foundations of Bayes Theorem

Author: Samuel Fredric Berg

Student ID: sb224sc

Date: 2026-05-19

Course: Deep Machine Learning 4DT908

## Assignment: Foundations of Bayes’ theorem

Fill out the blanks as per the instructions below.

This assignment uses type hints, so make sure to stick to those.

Whenever you need to fill in a blank, we used Python's ellipsis (`...`).

## Part 1 (A): Bayes' theorem with discrete random variables

Here, we assume a discrete prior $P(\theta)$, as well as a discrete probability distribution over a few i.i.d. observations.

The goal is to manually implement functionals for computing marginal and conditional likelihoods/probability densities.

You need to show that the posterior probability $P(\theta|Y)$ is a proper probability mass function.
Choose a different number of parameters and observations.

In [1]:
# Let's define our parameters theta and their probabilities (our prior belief):
# A handful of thetas is enough.
theta: list[int] = [-1, 0, 1, 2]
theta_probs: list[float] = [0.1, 0.2, 0.5, 0.2]

# Here are our observations Y (don't change!):
Y_obs = [0.5, 1.2]

# Instead of assuming some (parameterized) distribution,
# we hardcode the conditional likelihoods of Y given some theta.
# Note that P(Y|	heta) is a likelihood, so it does not represent
# (necessarily) a valid probability density (i.e., values for each
# 	heta do not necessarily have to sum to 1).
P_Y_given_theta: dict[int, dict[float, float]] = {
    -1: {0.5: 0.10, 1.2: 0.05},
    0: {0.5: 0.30, 1.2: 0.20},
    1: {0.5: 0.40, 1.2: 0.45},
    2: {0.5: 0.20, 1.2: 0.40},
}

### Define PMFs

For convenience, we define the PMFs for $\theta$ and $Y$ explicitly:

In [2]:
# Don't change!
def P_theta(val: float) -> float:
    assert val in theta
    idx = theta.index(val)
    return theta_probs[idx]

### Define Functions

for the likelihood $P(Y|\theta)$, the prior $P(\theta)$, and the evidence $P(Y)$.

Recall that the evidence:

$$
\begin{align}
    P(Y)&=\sum_i\,P(Y|\theta_i)\times P(\theta_i)\nonumber.
\end{align}
$$

In [3]:
# Likelihood, P(Y|	heta), now explicitly from our discrete definition:
def likelihood(Y: list[float], t: float) -> float:
    assert t in theta
    probs = P_Y_given_theta[t]
    out = 1.0
    for y in Y:
        out *= probs[y]
    return out


# The Evidence (in this assignment, it *is* computable):
def P_Y(Y: list[float]) -> float:
    return sum(likelihood(Y=Y, t=t) * P_theta(val=t) for t in theta)


# The posterior:
def P_theta_given_Y(t: float, Y: list[float]) -> float:
    return (likelihood(Y=Y, t=t) * P_theta(val=t)) / P_Y(Y=Y)

In [4]:
# Don't change!
posterior_probs = [round(P_theta_given_Y(t=t, Y=Y_obs), ndigits=5) for t in theta]
posterior_probs

[0.00422, 0.10127, 0.75949, 0.13502]

In [5]:
# Don't change! The result here needs to be ~1.0!
print(sum(posterior_probs))

1.0


## Part 1(B): Bayes' theorem with continuous random variables

-------------------------

Now, we change our model a bit.
Instead of assuming a small discrete set of possible values for $\theta$, we will assume that this parameter follows a standard normal distribution.

For our actual model, we will assume another normal distribution, where the standard deviation (scale) is fixed at $\frac{3}{2}$ and the mean is set to $\theta$: $N\sim(\mu=\theta,\sigma=\frac{3}{2})$.

We will re-use the previous observations.


The evidence, defined continuously:

$$
\begin{align}
    P(Y)=\int_{\theta}\,P(Y|t)\times P(t)\,d\theta\nonumber.
\end{align}
$$

In [6]:
from scipy.stats.distributions import norm


# Our prior:
def P_theta_continuous(val: float) -> float:
    # Use norm.pdf() to compute this.
    return norm.pdf(x=val, loc=0.0, scale=1.0).item()

In [7]:
import numpy as np
from scipy.integrate import quad
from typing import final

# Realistically, our bounds could be -10,10 (or similar), but
# scipy's quad allows to use infinity, so we'll use that, as
# it's also closer to how we would formulate this mathematically.
a, b = -np.inf, np.inf


# Our model prototype that takes a single scale parameter that
# will be held fixed for any subsequent likelihood computations.
@final
class Model:
    """Keep using this model class as-is, no need to change it."""

    def __init__(self, scale: float):
        self.scale = scale

    def likelihood(self, x: float, mean: float) -> float:
        return norm.pdf(x=x, loc=mean, scale=self.scale).item()


def likelihood_continuous(Y: list[float], t: float, model: Model) -> float:
    out = 1.0
    for y in Y:
        out *= model.likelihood(x=y, mean=t)
    return out


def P_Y_continuous(Y: list[float], model: Model) -> float:
    """Use quad() to integrate."""
    func = lambda t: likelihood_continuous(Y=Y, t=t, model=model) * P_theta_continuous(
        val=t
    )
    return quad(func=func, a=a, b=b)[0]


def P_theta_given_Y_continuous(
    t: float, Y: list[float], evidence: float, model: Model
) -> float:
    numerator = likelihood_continuous(Y=Y, t=t, model=model) * P_theta_continuous(val=t)
    return numerator / evidence


# The goal of this function is to assert that our posterior is
# a valid probability density that sums/integrates to 1.
def P_theta_given_Y_continuous_integral(Y: list[float], model: Model) -> float:
    evidence = P_Y_continuous(Y=Y, model=model)
    func = lambda t: P_theta_given_Y_continuous(
        Y=Y, t=t, model=model, evidence=evidence
    )
    return quad(func=func, a=a, b=b)[0]

Now we show the amount of evidence, as well as that our posterior integrates to $\approx1$:
Also, we show the amount of (log-)evidence:

In [8]:
# Don't change. Prints the log-evidence, as well as its integral (should be ~1.0).
from math import log

use_model = Model(scale=1.5)

log(P_Y_continuous(Y=Y_obs, model=use_model)), P_theta_given_Y_continuous_integral(
    Y=Y_obs, model=use_model
)

(-3.1912461104301157, 0.9999999999999991)

#### Find and use a better model

Recall that our observations were fixed at $[0.5, 1.2]$ and we assumed our model would be a normal distribution with standard deviation $\sigma=\frac{3}{2}$.

In [9]:
Y_obs_arr = np.array(Y_obs)
Y_obs_arr.std().item(), Y_obs_arr.mean().item()

(0.35, 0.85)

However, we know that the standard deviation should likely be smaller to accommodate our data better.
What we want to show here, is that a better model (here: same as previous but with a fixed standard deviation closer to $0.35$) produces a larger **evidence**.

In [10]:
from scipy.optimize import minimize_scalar

# Use 'minimize_scalar' to find some optimal solution
optimal_scale = minimize_scalar(
    lambda s: -np.log(P_Y_continuous(Y=Y_obs, model=Model(scale=s))),
    bounds=(1e-3, 5.0),
    method="bounded",
).x  # <- fill in the blank here

In [11]:
# Don't change! Prints the log-evidence, as well as its integral (should be ~1.0).
better_model = Model(scale=optimal_scale)

log(P_Y_continuous(Y=Y_obs, model=better_model)), P_theta_given_Y_continuous_integral(
    Y=Y_obs, model=better_model
)

(-2.360448901822983, 0.9999999999999998)

## Short evaluation (write 1-2 sentences per):

1. How has the (log-)evidence changed using the optimal scale?
2. In Bayesian terms, what does this result mean?

**Answers**:

1. The log-evidence increased (became less negative) after using the optimized scale, which means the model with that scale assigns higher probability to the observed data.
2. In Bayesian terms, the optimized model provides a better explanation of the observations under the prior-and-likelihood setup, so the posterior is supported by stronger evidence.
